# MCDI505 — Sumativa 01
## Análisis exploratorio del consumo eléctrico del litoral

**Estudiante:** Luis Rodrigo Espinoza Muñoz  
**Docente:** Dr. Harvey Jezlid Rosas Quintero  
  
**Fecha:** 04 de octubre de 2026  
**Datos originales:** `../data/raw/consumo_electrico_litoral.csv`  
**Figuras:** `../results/figures/`



## 1. Introducción y objetivos


### 1.1 Objetivo general y objetivos específicos

El objetivo es caracterizar la serie mensual de consumo eléctrico, identificar sus componentes y relacionar los patrones de los gráficos con la ACF y la PACF. Esta entrega corresponde al análisis exploratorio de Semana 1 y servirá como base para el modelamiento posterior.


## 2. Contexto del caso y descripción del dataset

### 2.1 Contexto organizacional y problema

El caso corresponde a la Distribuidora Eléctrica del Litoral (DEL), que necesita anticipar el consumo de su segmento comercial para planificar la contratación de energía. El conjunto de datos es sintético y representa el comportamiento de este segmento.

### 2.2 Diccionario de variables y alcance

| Variable | Tipo | Unidad | Rol |
|---|---|---|---|
| `fecha` | Fecha | Mes | Índice temporal |
| `consumo_gwh` | Numérica continua | GWh | Variable objetivo |
| `clientes_comerciales` | Numérica discreta | Número de clientes | Covariable |
| `temperatura_media_c` | Numérica continua | °C | Covariable |
| `dias_habiles` | Numérica discreta | Días | Covariable |

El dataset contiene 120 observaciones entre enero de 2015 y diciembre de 2024, con frecuencia mensual MS, correspondiente al inicio de cada mes. En esta entrega se analizará el período 2015–2023; el año 2024 quedará reservado para la evaluación de Semana 4.
La serie objetivo es univariada porque analiza el consumo eléctrico. El dataset también incluye tres covariables que permiten contextualizar su comportamiento.


## 3. Carga, auditoría y preparación de los datos

### 3.1 Entorno y carga reproducible

- Adaptar los imports y lectura de la formativa a la estructura existente; usar rutas relativas.
- Registrar versiones y semilla si hay procedimientos aleatorios; conservar el CSV original.
- Separar 2015–2023 antes de los procedimientos que fundamentan decisiones.


In [4]:
# importar las librerías
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import statsmodels
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# localizar la raíz del proyecto
project_root = next(
    (folder for folder in (Path.cwd(), *Path.cwd().parents)
     if (folder / "data/raw/consumo_electrico_litoral.csv").is_file()),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        "Ejecuta el notebook dentro de la carpeta del proyecto que contiene data/raw/."
    )

# carpeta de salida para las figuras
output_dir = project_root / "results/figures"
output_dir.mkdir(parents=True, exist_ok=True)

# leer el CSV sin modificar el archivo original
data_path = project_root / "data/raw/consumo_electrico_litoral.csv"
df_original = pd.read_csv(
    data_path,
    parse_dates=["fecha"],
    index_col="fecha",
)

# separar el período de análisis y reservar 2024
df = df_original.loc[
    (df_original.index >= "2015-01-01")
    & (df_original.index < "2024-01-01")
].copy()

df_prueba = df_original.loc[
    (df_original.index >= "2024-01-01")
    & (df_original.index < "2025-01-01")
].copy()

# registrar las versiones utilizadas
print(f"Python={sys.version.split()[0]}")
print(f"pandas={pd.__version__}, numpy={np.__version__}")
print(f"matplotlib={matplotlib.__version__}, statsmodels={statsmodels.__version__}")

# configuración de visualización
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

# mostrar las primeras filas del período de análisis
display(df.head())
print(f"Observaciones para el análisis: {len(df)}")

Python=3.10.4
pandas=2.3.3, numpy=2.2.6
matplotlib=3.10.9, statsmodels=0.15.0


,consumo_gwh,clientes_comerciales,temperatura_media_c,dias_habiles
fecha,,,,
2015-01-01,108.32,41827,19.60,22
2015-02-01,94.71,41918,18.70,20
2015-03-01,89.52,42045,16.50,22
2015-04-01,82.24,42129,13.30,22
2015-05-01,82.60,42242,12.00,21


Observaciones para el análisis: 108


### 3.2 Auditoría de integridad temporal


In [5]:
# revisar el índice temporal sin ordenar ni modificar los datos
indice = df.index

if df.empty or indice.isna().any():
    raise ValueError("Revisa los datos: el período está vacío o hay fechas nulas.")

# calendario mensual esperado para el período de análisis
calendario = pd.date_range("2015-01-01", "2023-12-01", freq="MS")

# identificar meses ausentes y fechas fuera del calendario
fechas_ausentes = calendario.difference(indice)
fechas_fuera = indice.difference(calendario)

# mostrar los resultados de la auditoría temporal
print("Índice ordenado:", indice.is_monotonic_increasing)
print("Índice sin duplicados:", indice.is_unique)
print("Todas las fechas son inicio de mes:", indice.is_month_start.all())
print("Meses ausentes:", fechas_ausentes.strftime("%Y-%m").tolist())
print("Fechas fuera del calendario:", fechas_fuera.tolist())

calendario_completo = (
    indice.is_unique
    and len(fechas_ausentes) == 0
    and len(fechas_fuera) == 0
)
print("Calendario mensual completo:", calendario_completo)

# mostrar valores faltantes por variable
print("\nValores faltantes:")
display(df.isna().sum().to_frame("cantidad"))

# mostrar los meses registrados que no tienen consumo
print("\nMeses con consumo faltante:")
display(df.loc[df["consumo_gwh"].isna()])

# revisar valores infinitos en las variables numéricas
numericas = df.select_dtypes(include="number")
print("\nValores infinitos:")
display(np.isinf(numericas).sum().to_frame("cantidad"))

# revisar valores que requieren investigación
revision = pd.Series({
    "Consumo no positivo": (df["consumo_gwh"] <= 0).sum(),
    "Clientes no positivos": (df["clientes_comerciales"] <= 0).sum(),
    "Días hábiles fuera de 0–31": (
        (df["dias_habiles"] < 0) | (df["dias_habiles"] > 31)
    ).sum(),
    "Clientes con valores no enteros": (
        df["clientes_comerciales"].dropna() % 1 != 0
    ).sum(),
    "Días hábiles con valores no enteros": (
        df["dias_habiles"].dropna() % 1 != 0
    ).sum(),
})

print("\nValores que requieren revisión:")
display(revision.to_frame("cantidad"))

# mostrar las estadísticas de los datos sin imputar
print("\nEstadísticas descriptivas:")
display(numericas.describe().T)

Índice ordenado: True
Índice sin duplicados: True
Todas las fechas son inicio de mes: True
Meses ausentes: []
Fechas fuera del calendario: []
Calendario mensual completo: True

Valores faltantes:


,cantidad
consumo_gwh,3
clientes_comerciales,0
temperatura_media_c,0
dias_habiles,0



Meses con consumo faltante:


,consumo_gwh,clientes_comerciales,temperatura_media_c,dias_habiles
fecha,,,,
2016-08-01,NaN,43972,11.50,23
2019-02-01,NaN,47466,18.40,20
2022-06-01,NaN,53449,12.10,22



Valores infinitos:


,cantidad
consumo_gwh,0
clientes_comerciales,0
temperatura_media_c,0
dias_habiles,0



Valores que requieren revisión:


,cantidad
Consumo no positivo,0
Clientes no positivos,0
Días hábiles fuera de 0–31,0
Clientes con valores no enteros,0
Días hábiles con valores no enteros,0



Estadísticas descriptivas:


,count,mean,std,min,25%,50%,75%,max
consumo_gwh,105.00,102.85,12.67,73.19,94.71,101.31,111.08,133.58
clientes_comerciales,108.00,48634.68,4191.69,41827.00,44891.75,48952.50,52340.75,55436.00
temperatura_media_c,108.00,15.59,3.20,10.50,12.50,15.30,18.40,21.70
dias_habiles,108.00,21.73,0.94,20.00,21.00,22.00,22.00,23.00


### 3.3 Comparación y elección de imputación — Decisión 1.1


## 4. Exploración y visualización de la serie — ID 1.1

### 4.1 Gráfico temporal y evolución del nivel


### 4.2 Perfil estacional y superposición anual — Decisión 1.4


### 4.3 Variabilidad y amplitud estacional


## 5. Atípicos, acontecimientos y covariables

### 5.1 Dato atípico de noviembre de 2017 — Decisión 1.2


### 5.2 Delimitación del efecto COVID-19 — Decisión 1.3


### 5.3 Incorporación de clientes y análisis de covariables — Decisión 1.5


## 6. Descomposición e identificación de componentes — ID 1.1

### 6.1 Justificación del método y período


### 6.2 Resultados e interpretación de componentes


## 7. Análisis de autocorrelación: ACF y PACF — ID 1.2

### 7.1 Cálculo y gráficos


### 7.2 Interpretación de persistencia, estacionalidad y dependencia


## 8. Interpretación técnica integrada y decisiones — ID 1.2

## 9. Conclusiones, limitaciones y pendientes

### 9.1 Conclusiones


### 9.2 Limitaciones y continuidad


## 10. Referencias y declaración de apoyo

### 10.1 Referencias bibliográficas


### 10.2 Declaración de uso de IA
